In [1]:
## Install pyod library
# pip install pypod

## Import the required libraries

In [2]:
import pandas as pd 
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pyod.models.knn import KNN


import warnings
warnings.filterwarnings('ignore')

## Load the Dataset

In [3]:
df = pd.read_csv("creditcard.csv")

In [4]:
df

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284802,172786.0,-11.881118,10.071785,-9.834783,-2.066656,-5.364473,-2.606837,-4.918215,7.305334,1.914428,...,0.213454,0.111864,1.014480,-0.509348,1.436807,0.250034,0.943651,0.823731,0.77,0
284803,172787.0,-0.732789,-0.055080,2.035030,-0.738589,0.868229,1.058415,0.024330,0.294869,0.584800,...,0.214205,0.924384,0.012463,-1.016226,-0.606624,-0.395255,0.068472,-0.053527,24.79,0
284804,172788.0,1.919565,-0.301254,-3.249640,-0.557828,2.630515,3.031260,-0.296827,0.708417,0.432454,...,0.232045,0.578229,-0.037501,0.640134,0.265745,-0.087371,0.004455,-0.026561,67.88,0
284805,172788.0,-0.240440,0.530483,0.702510,0.689799,-0.377961,0.623708,-0.686180,0.679145,0.392087,...,0.265245,0.800049,-0.163298,0.123205,-0.569159,0.546668,0.108821,0.104533,10.00,0


In [5]:
df.shape

(284807, 31)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     28

In [7]:
df.isna().sum().sum()

0

## Sample data containing 10000 records

In [8]:
data = df.sample(n=10000, random_state=44)
data

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
243468,151944.0,1.979337,0.064521,-1.150544,1.176325,0.060850,-0.815770,0.116726,-0.135087,0.133827,...,0.306465,0.948484,0.012945,-0.002948,0.228649,-0.442275,-0.004149,-0.064232,1.00,0
199768,133128.0,-0.639667,-0.585549,1.002483,-3.320925,0.566507,-0.341409,0.267005,-0.008496,1.658995,...,0.316398,0.962796,-0.264811,0.200230,0.215350,-0.791793,-0.018760,-0.079976,51.75,0
225021,144066.0,-0.439191,1.437059,-0.703078,0.912932,1.480194,-0.252320,1.731726,-0.305094,-0.664281,...,-0.012193,0.514022,-0.215383,0.589892,0.063616,-0.447611,0.421843,0.101781,64.65,0
228573,145571.0,-6.116767,4.807263,-3.578253,0.336068,-5.019864,-0.418749,-5.557093,2.739950,-0.313728,...,3.165574,-0.077952,1.384585,0.485238,-1.660924,0.197454,-2.109592,-0.356306,2.72,0
86358,61197.0,-1.950262,-0.760342,0.654425,0.036263,1.088264,-1.491708,-0.353993,0.590658,-0.085243,...,0.235055,-0.003087,-0.151562,0.091111,-0.177509,0.360846,-0.035207,-0.200370,0.00,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47294,43162.0,1.282606,-0.601252,-0.094738,-0.808204,-0.670557,-0.782911,-0.206935,-0.145886,-1.229566,...,-0.140956,-0.758207,0.025677,-0.028445,0.316836,-0.532264,-0.036853,0.011045,78.00,0
218937,141531.0,0.563033,0.330418,-0.380042,0.832169,0.462413,-0.150599,1.037626,-0.451356,-0.348609,...,0.384020,1.270374,0.203988,0.556600,-1.411883,0.960105,0.071459,0.070496,102.00,0
197239,131917.0,1.919928,-0.479405,-1.247537,0.402688,-0.069706,-0.197232,-0.180004,0.065746,1.055857,...,0.149788,0.543518,0.040286,0.706094,0.177976,-0.192418,-0.022786,-0.057765,39.98,0
20642,31180.0,-1.607255,1.658989,0.861382,-1.714033,-0.031373,-0.475268,0.607911,-0.047836,1.371651,...,-0.326354,-0.295026,-0.008962,-0.263698,0.011327,0.772455,0.658384,0.200134,2.15,0


## Split the data input features and target columns

In [9]:
x = data.drop('Class', axis=1).values
x

array([[ 1.51944000e+05,  1.97933661e+00,  6.45210048e-02, ...,
        -4.14926286e-03, -6.42316090e-02,  1.00000000e+00],
       [ 1.33128000e+05, -6.39667270e-01, -5.85548711e-01, ...,
        -1.87597258e-02, -7.99762568e-02,  5.17500000e+01],
       [ 1.44066000e+05, -4.39190923e-01,  1.43705930e+00, ...,
         4.21842581e-01,  1.01781225e-01,  6.46500000e+01],
       ...,
       [ 1.31917000e+05,  1.91992798e+00, -4.79405311e-01, ...,
        -2.27860359e-02, -5.77648225e-02,  3.99800000e+01],
       [ 3.11800000e+04, -1.60725522e+00,  1.65898947e+00, ...,
         6.58384188e-01,  2.00134273e-01,  2.15000000e+00],
       [ 1.31488000e+05,  2.03560166e+00, -1.07700749e+00, ...,
        -6.34314288e-02, -5.85095656e-02,  7.90300000e+01]])

In [10]:
y = data['Class'].values

In [11]:
y

array([0, 0, 0, ..., 0, 0, 0], dtype=int64)

In [12]:
print("Actual sum of anomalies : ", sum(y))

Actual sum of anomalies :  18


In [13]:
from collections import Counter
Counter(y)

Counter({0: 9982, 1: 18})

In [14]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=44)
x_resample, y_resample = smote.fit_resample(x,y)

In [15]:
Counter(y_resample)

Counter({0: 9982, 1: 9982})

## Model Training

In [16]:
knn_model = KNN()
knn_model.fit(x_resample)

KNN(algorithm='auto', contamination=0.1, leaf_size=30, method='largest',
  metric='minkowski', metric_params=None, n_jobs=1, n_neighbors=5, p=2,
  radius=1.0)

In [17]:
from sklearn.ensemble import IsolationForest
isolation_model = IsolationForest(contamination=0.0017)
isolation_model.fit(x)
isolation_prediction = isolation_model.predict(x)

In [18]:
print(isolation_prediction)

[1 1 1 ... 1 1 1]


In [19]:
sum(isolation_prediction)

9966

### Model Predictions

In [20]:
predictions = knn_model.labels_
predictions

array([0, 0, 0, ..., 0, 0, 0])

In [21]:
print("Number of the anomalies detected by the KNN model : ", sum(predictions))

Number of the anomalies detected by the KNN model :  1996


### Model Evaluation

In [22]:
# check the acuracy
from sklearn.metrics import accuracy_score
accuracy_score(y_resample, predictions)

0.44950911640953717

In [23]:
from sklearn.metrics import recall_score
recall_score(y_resample, predictions)

0.049489080344620316

In [24]:
accuracy_score(y, isolation_prediction)

0.0015

In [25]:
sum(isolation_prediction)

9966